In [ ]:
import os
import logging
import torch
from torch.utils.data import Dataset, DataLoader
from torch import optim, nn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score, precision_score, recall_score, f1_score, accuracy_score, roc_auc_score
from datetime import datetime
import random

def set_seed(seed):
    random.seed(seed)  
    np.random.seed(seed)  
    torch.manual_seed(seed)  
    torch.cuda.manual_seed(seed) 
    torch.cuda.manual_seed_all(seed)  
    torch.backends.cudnn.deterministic = True  
    torch.backends.cudnn.benchmark = False 

set_seed(42)

# 超参数设置

In [ ]:
class Config:
    def __init__(self):
        self.data_path = '/data/'
        self.data_file_name = 'data.csv'
        self.train_ratio = 0.8  
        self.pattern = 'train'  
        self.scaler_x = StandardScaler()
        self.scaler_y = StandardScaler()
        self.time_delay = 5  
        self.forward_step = 1  # predictive time length
        self.batch_size = 32
        self.learning_rate = 1e-5
        self.epoches = 200
        self.hidden_size = 256  
        self.include_target_in_x = True 
        self.input_size = 10  
        self.num_layers = 2 
        self.dropout = 0.25  
        self.weight_decay = 0.25  
        self.device = torch.device(f"cuda:{0}" if torch.cuda.is_available() else "cpu")

# 数据加载与处理

In [ ]:
def data_process(args):
    all_data = pd.read_csv(args.data_path + args.data_file_name, header=0, index_col=0).values.astype(np.float32)
    train_size = int(len(all_data) * args.train_ratio)
    or_train = all_data[:train_size]
    or_test = all_data[train_size:]

    train_x_raw = or_train[:, :-5]
    test_x_raw = or_test[:, :-5]
    if args.include_target_in_x:
        train_x_raw = np.concatenate([train_x_raw, or_train[:, -1].reshape(-1, 1)], axis=1)
        test_x_raw = np.concatenate([test_x_raw, or_test[:, -1].reshape(-1, 1)], axis=1)

    args.scaler_x.fit(train_x_raw)
    args.scaler_y.fit(or_train[:, -1].reshape(-1, 1))

    scale_train_x = args.scaler_x.transform(train_x_raw)
    scale_train_y = args.scaler_y.transform(or_train[:, -1].reshape(-1, 1)).reshape(-1)
    scale_test_x = args.scaler_x.transform(test_x_raw)
    scale_test_y = args.scaler_y.transform(or_test[:, -1].reshape(-1, 1)).reshape(-1)

    return (scale_train_x, scale_train_y), (scale_test_x, scale_test_y)


class MyDataset(Dataset):
    def __init__(self, scale_x, scale_y, look_back=30, pred_len=5):
        super(MyDataset, self).__init__()
        self.sca_x = scale_x
        self.sca_y = scale_y
        self.lag = look_back
        self.step = pred_len

    def __getitem__(self, item):
        seq_x = self.sca_x[item:item + self.lag, :]
        pred = self.sca_y[item + self.lag + self.step - 1]

        return seq_x, pred

    def __len__(self):
        return self.sca_y.shape[0] - self.lag - self.step + 1


def collate_fn(batch):
    data_x, data_y = zip(*batch)
    return torch.tensor(np.array(data_x)), torch.tensor(np.array(data_y)).squeeze()


def load_data(args):
    train_data, test_data = data_process(args)
    train_data_set = MyDataset(*train_data, args.time_delay, args.forward_step)
    train_data_loader = DataLoader(train_data_set, batch_size=args.batch_size, collate_fn=collate_fn)
    test_data_set = MyDataset(*test_data, args.time_delay, args.forward_step)
    test_data_loader = DataLoader(test_data_set, args.batch_size, collate_fn=collate_fn)
    return train_data_loader, test_data_loader

# 模型构建

In [ ]:
class lstm_model(nn.Module):
    def __init__(self, seq_len, input_dim, hidden_dim, num_layers, dropout):
        super(lstm_model, self).__init__()

        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout)

        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim*4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim*4, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x) 
        out = out[:, -1, :]  
        output = self.fc(out)
        return output

# 绘图函数

In [ ]:
def draw_result(x, y, r2, plot_title=None, output_dir=None):
    plt.figure(figsize=(10, 5))
    plt.plot(x, linestyle='-', color='black', label='Raw Data')
    plt.plot(y, linestyle='--', color='red', label=f'Prediction Data \n (R$^2$ = {r2:.4f})')
    plt.xlabel('Time')
    plt.ylabel('Value')
    plt.grid(True)
    plt.legend()
    if plot_title:
        plt.title(plot_title)
        filename = f'{plot_title}.svg'
        if output_dir:
            plt.savefig(os.path.join(output_dir, filename), bbox_inches='tight')
        else:
            plt.savefig(filename, bbox_inches='tight')
        plt.show()
    else: 
        plt.show()

# 训练与测试函数

In [ ]:
def get_output_dir(args):
    data_tag = os.path.splitext(args.data_file_name)[0]
    name = (
        f"{data_tag}_td{args.time_delay}_fs{args.forward_step}_bs{args.batch_size}_"
        f"lr{args.learning_rate}_hs{args.hidden_size}_nl{args.num_layers}"
    )
    out_dir = os.path.join(args.data_path, "outputs", name)
    os.makedirs(out_dir, exist_ok=True)
    return out_dir


def setup_logger(log_path):
    logger = logging.getLogger(log_path)
    logger.setLevel(logging.INFO)
    logger.handlers = []
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

    file_handler = logging.FileHandler(log_path)
    file_handler.setLevel(logging.INFO)
    file_handler.setFormatter(formatter)

    stream_handler = logging.StreamHandler()
    stream_handler.setLevel(logging.INFO)
    stream_handler.setFormatter(formatter)

    logger.addHandler(file_handler)
    logger.addHandler(stream_handler)
    return logger


def model_train(model, train_data_set, test_data_set, criterion, optimizer, args, output_dir):
    loss_train_epoch = []
    loss_test_epoch = []

    log_path = os.path.join(output_dir, 'train_experiment.log')
    logger = setup_logger(log_path)
    

    with open(os.path.join(output_dir, 'training_loss_log.txt'), 'w') as f:
        f.write("Epoch,Train Loss,Test Loss\n")  
        f.flush()  

        plt.ion()
        for epoch in range(args.epoches):
            loss_train_iteration = 0.0
            loss_test_iteration = 0.0
            train_sample_num = 0
            test_sample_num = 0
            
            model.train()
            for _, (train_x, train_y) in enumerate(train_data_set):
                train_x, train_y = train_x.to(args.device), train_y.to(args.device)
                y_hat = model.forward(train_x)
                loss = criterion(y_hat.squeeze(), train_y)
                loss.backward()
                optimizer.step()
                loss_train_iteration += loss.item()
                train_sample_num += train_x.size(0)

            model.eval()
            with torch.no_grad():
                for _, (test_x, test_y) in enumerate(test_data_set):
                    test_x, test_y = test_x.to(args.device), test_y.to(args.device)
                    y_hat = model.forward(test_x)
                    loss_mse = criterion(y_hat.squeeze(), test_y).item()
                    loss_test_iteration += loss_mse
                    test_sample_num += test_x.size(0)

            train_loss = loss_train_iteration / train_sample_num
            test_loss = loss_test_iteration / test_sample_num
            loss_train_epoch.append(train_loss)
            loss_test_epoch.append(test_loss)

            lr = optimizer.param_groups[0]['lr']
            logger.info(
                f"Epoch {epoch + 1}/{args.epoches}, Train Loss: {train_loss:.4f}, "
                f"Test Loss: {test_loss:.4f}, Lr: {lr:.6f}"
            )

            f.write(f"{epoch + 1},{train_loss:.6f},{test_loss:.6f}\n")
            f.flush()

            if (epoch + 1) % 50 == 0:
                for param_group in optimizer.param_groups:
                    param_group['lr'] *= 0.5
                logger.info(f"LR decayed to {optimizer.param_groups[0]['lr']:.6f}")

        plt.figure()
        plt.plot(range(1, epoch + 2), loss_train_epoch, label='Train Loss')
        plt.plot(range(1, epoch + 2), loss_test_epoch, label='Test Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Training and Testing Loss per Epoch')
        plt.legend()
        plt.savefig(os.path.join(output_dir, 'final_epoch_loss.svg'))
        plt.show()

    logger.info('Finish training')
    return model


def model_val(model, data_set, args, plot_title=None, output_dir=None, log_file_name=None):
    criter_mse = nn.MSELoss()
    criter_mae = nn.L1Loss()
    pre_scale_y = []
    true_scale_y = []

    logger = None
    if output_dir and log_file_name:
        log_path = os.path.join(output_dir, log_file_name)
        logger = setup_logger(log_path)
    
    model.eval()
    for _, (train_x, train_y) in enumerate(data_set):
        train_x, train_y = train_x.to(args.device), train_y.to(args.device)
        y_hat = model.forward(train_x)
        pre_values = np.atleast_1d(np.array(y_hat.cpu().squeeze().data))
        true_values = np.atleast_1d(np.array(train_y.cpu().data))
        pre_scale_y.extend(pre_values.tolist())
        true_scale_y.extend(true_values.tolist())
    
    pre_scale_y = np.array(pre_scale_y).reshape(-1, 1)
    true_scale_y = np.array(true_scale_y).reshape(-1, 1)
    pre_y = args.scaler_y.inverse_transform(pre_scale_y).reshape(-1)
    true_y = args.scaler_y.inverse_transform(true_scale_y).reshape(-1)
    
    result_df = pd.DataFrame({'True_Y': true_y, 'Predicted_Y': pre_y})
    if output_dir:
        result_df.to_excel(os.path.join(output_dir, 'model_predictions.xlsx'), index=False)
    else:
        result_df.to_excel('model_predictions.xlsx', index=False)

    loss_mse = criter_mse(torch.tensor(true_y, dtype=torch.float32),
                          torch.tensor(pre_y, dtype=torch.float32))
    loss_mae = criter_mae(torch.tensor(true_y, dtype=torch.float32),
                          torch.tensor(pre_y, dtype=torch.float32))

    true_y_tensor = torch.tensor(true_y, dtype=torch.float32)
    pre_y_tensor = torch.tensor(pre_y, dtype=torch.float32)
    loss_mape = torch.mean(torch.abs((true_y_tensor - pre_y_tensor) / (true_y_tensor + 1e-8))) * 100
    r2 = r2_score(true_y, pre_y)

    threshold = 25

    true_class = (true_y >= threshold).astype(int)
    pred_class = (pre_y >= threshold).astype(int)

    try:
        prec = precision_score(true_class, pred_class, zero_division=0)
        rec = recall_score(true_class, pred_class, zero_division=0)
        f1 = f1_score(true_class, pred_class, zero_division=0)
    except:
        prec, rec, f1 = 0, 0, 0


    if logger:
        logger.info(f"MSE: {loss_mse}")
        logger.info(f"MAE: {loss_mae}")
        logger.info(f"MAPE(%): {loss_mape}")
        logger.info(f"R2: {r2}")
        logger.info(f"Precision: {prec}")
        logger.info(f"Recall: {rec}")
        logger.info(f"F1 Score: {f1}")
    else:
        print('MSE', loss_mse)
        print('MAE', loss_mae)
        print('MAPE(%)', loss_mape)
        print('R2', r2)
        print('Precision', prec)
        print('Recall', rec)
        print('F1 Score', f1)


    draw_result(true_y, pre_y, r2, plot_title, output_dir=output_dir)

# 主函数

In [ ]:
args = Config()
train_loader, test_loader = load_data(args)

output_dir = get_output_dir(args)

model = lstm_model(args.time_delay, args.input_size, args.hidden_size, args.num_layers, args.dropout).to(args.device)
total_param = sum([param.nelement() for param in model.parameters()])
print('Number of parameter: %d' % (total_param))
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=args.learning_rate, weight_decay=args.weight_decay)

if args.pattern == 'train':
    model = model_train(model, train_loader, test_loader, criterion, optimizer, args, output_dir)
    torch.save(model.state_dict(), os.path.join(output_dir, 'LSTM_ATT.pth'))
else:
    model.load_state_dict(torch.load(os.path.join(output_dir, 'LSTM_ATT.pth')))
    model_val(model, test_loader, args, plot_title='Test Data Fitting Results', output_dir=output_dir, log_file_name='test_experiment.log')

# 绘图

## 训练集

In [ ]:
model_val(model, train_loader, args, plot_title='Train Data Fitting Results', output_dir=output_dir, log_file_name='train_experiment.log')

## 测试集

In [ ]:
model_val(model, test_loader, args, plot_title='Test Data Fitting Results', output_dir=output_dir, log_file_name='test_experiment.log')